# 06. 단계적 공개와 설계 비교

단계적 공개는 문서 ID와 제목만 먼저 보고, 질문에 필요한 문서를 고른 뒤, 그 문서의 자세한 본문만 모델에 전달하는 방식입니다.

## 실행 환경 준비

실습 파일은 `labs/part2/ch08-context-engineering/notebooks/06-단계적-공개와-설계-비교.ipynb`입니다. 새 커널에서 시작해 첫 코드 셀로 패키지 버전을 맞춘 뒤, 설명과 코드를 위에서부터 순서대로 실행합니다. `.env`는 현재 작업 폴더의 상위 경로에서도 찾습니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
import os

from dotenv import find_dotenv, load_dotenv

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print(".env와 API 키 확인 완료")

.env와 API 키 확인 완료


### 모델 객체를 만듭니다

`.env` 확인이 끝난 뒤 모델 이름과 출력 제한을 지정합니다. 이 셀은 아직 API에 질문을 보내지 않습니다.

In [3]:
# init_chat_model은 provider:model 형식으로 채팅 모델 객체를 만듭니다.
# model은 이후 셀에서 invoke()를 호출할 때 사용합니다.
from langchain.chat_models import init_chat_model

# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=300,
)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


### Ch 7 실행 기록을 확인합니다

Ch 7의 프롬프트 A/B 기록을 읽고 필수 필드와 파일 지문을 확인합니다. 이전 실행 수치를 현재 실행 결과와 직접 비교하지는 않습니다.

In [4]:
from hashlib import sha256
import json
from pathlib import Path

course_root = next((path for path in [Path.cwd(), *Path.cwd().parents]
                    if (path / "labs/part2/ch07-prompt-engineering").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("실습 묶음의 labs/part2 폴더를 찾지 못했습니다.")
ch07_relative = Path("labs/part2/ch07-prompt-engineering/runs/prompt-ab-continuity.json")
ch07_path = course_root / ch07_relative
if not ch07_path.exists():
    raise FileNotFoundError("Ch 7의 08 노트북을 먼저 실행하세요.")
ch07_record = json.loads(ch07_path.read_text(encoding="utf-8"))
ch07_sha256 = sha256(ch07_path.read_bytes()).hexdigest()
required = {"run_id", "run_date", "source_chapter", "source_record", "settings", "versions"}
assert required <= ch07_record.keys()
assert ch07_record["settings"]["pricing_usd_per_million"] == {
    "ordinary_input": 0.75, "cached_input": 0.075, "output": 4.50}
print("Ch 7 기록:", ch07_record["run_id"], ch07_sha256[:12])

Ch 7 기록: ch07-prompt-ab-20260909T175111163232Z a798a5763a5e


## 2. 목록과 원문을 분리합니다

목록에는 여러 문서의 ID와 제목만 넣어 선택을 돕습니다. 답에 필요한 근거는 선택한 문서의 원문에서 가져옵니다.

In [5]:
# documents는 ID·제목·원문을 가진 문서 목록입니다.
documents = [
    {"id": "E01-p2", "title": "이용 시간", "text": "월요일 휴관"},
    {"id": "E01-p3", "title": "대출 규정",
     "text": "회원 1명당 최대 7권, 대출 기간 14일. 예약이 없으면 1회 7일 연장."},
    {"id": "E01-p5", "title": "주차 안내", "text": "회원은 최초 2시간 무료입니다."},
]
print("전체 문서 수:", len(documents))

전체 문서 수: 3


### 먼저 제목 목록만 공개합니다

원문 전체를 읽기 전에 ID와 제목만 복사합니다. 반복문에서 만든 `item`을 `catalog`에 하나씩 추가합니다.

In [6]:
# catalog에는 선택에 필요한 두 필드만 남깁니다.
catalog = []
for document in documents:
    item = {"id": document["id"], "title": document["title"]}
    catalog.append(item)

print(catalog)

[{'id': 'E01-p2', 'title': '이용 시간'}, {'id': 'E01-p3', 'title': '대출 규정'}, {'id': 'E01-p5', 'title': '주차 안내'}]


## 3. 필요한 ID의 원문만 공개합니다

이 예제에서는 학생이 `E01-p3`을 직접 선택합니다.

In [7]:
# selected_id는 질문에 맞춰 고른 문서 ID입니다.
# 반복문은 ID가 같은 문서를 찾으면 selected에 저장하고 중단합니다.
selected_id = "E01-p3"
selected = None
for document in documents:
    if document["id"] == selected_id:
        selected = document
        break

if selected is None:
    raise ValueError("선택한 문서를 찾지 못했습니다.")

print("선택한 제목:", selected["title"])

선택한 제목: 대출 규정


### 세 가지 공개 방식의 길이를 비교합니다

전체 원문, 선택한 원문, 사람이 작성한 압축문을 같은 문자 수 기준으로 비교합니다.

In [8]:
# full_lines를 만든 뒤 join으로 출처가 붙은 전체 원문을 연결합니다.
full_lines = []
for document in documents:
    line = f"[{document['id']}] {document['text']}"
    full_lines.append(line)

full = "\n".join(full_lines)
compressed = "1명당 최대 7권, 14일 대출, 예약 없으면 1회 7일 연장 [E01-p3]"
print("full 문자 수:", len(full))
print("selected 문자 수:", len(selected["text"]))
print("compressed 문자 수:", len(compressed))

full 문자 수: 94
selected 문자 수: 42
compressed 문자 수: 43


## 4. 세 설계를 같은 조건으로 실행합니다

전체 원문, 선택 원문, 압축문을 이름별로 준비합니다. 선택 원문에도 같은 출처 ID를 붙여 세 입력의 표기 조건을 맞춥니다.

In [9]:
# design_contexts의 값만 다르고 질문과 지시는 다음 셀에서 고정합니다.
selected_context = f"[{selected['id']}] {selected['text']}"
design_contexts = {
    "full": full,
    "selected": selected_context,
    "compressed": compressed,
}
for name, context in design_contexts.items():
    print(name, "문자 수:", len(context))

full 문자 수: 94
selected 문자 수: 51
compressed 문자 수: 43


### 같은 질문과 지시로 세 번 호출합니다

압축문은 앞에서 사람이 작성한 값이므로 별도의 압축 모델 호출 비용은 없습니다. 여기서 측정하는 사용량은 세 답변 호출의 입력 토큰입니다.

In [10]:
# Ch 6과 같은 기준일 단가로 OpenAI 호출 비용을 계산합니다.
def estimate_openai_cost(usage):
    input_tokens = usage.get("input_tokens")
    output_tokens = usage.get("output_tokens")
    if input_tokens is None or output_tokens is None:
        return None
    cache_read = (usage.get("input_token_details") or {}).get("cache_read") or 0
    normal_input = input_tokens - cache_read
    return (normal_input * 0.75 + cache_read * 0.075 + output_tokens * 4.50) / 1_000_000

In [11]:
# 모든 설계에 같은 instruction과 question을 사용합니다.
import time

instruction = "근거의 조건·예외·출처를 빠뜨리지 마세요."
question = "대출 권수, 기간, 연장 조건과 출처를 알려 주세요."
design_results = {}
for name, context in design_contexts.items():
    messages = [("system", instruction), ("user", f"근거: {context}\n질문: {question}")]
    started = time.perf_counter()
    response = model.invoke(messages)
    latency_ms = round((time.perf_counter() - started) * 1000, 1)
    usage = response.usage_metadata
    if usage is None:
        raise RuntimeError("모델 사용량 메타데이터가 없습니다.")
    cache_read = (usage.get("input_token_details") or {}).get("cache_read") or 0
    design_results[name] = {"response": response, "usage": usage,
                            "latency_ms": latency_ms, "cache_read_tokens": cache_read,
                            "estimated_cost_usd": round(estimate_openai_cost(usage), 8)}

### 실제 출력과 입력 토큰을 관찰합니다

설계 이름 아래에 입력 토큰과 답변 원문을 함께 출력합니다. 세 응답의 표현은 실행마다 달라질 수 있습니다.

In [12]:
# 토큰·지연·비용과 답변 원문을 설계별로 확인합니다.
for name, result in design_results.items():
    usage = result["usage"]
    print(f"\n[{name}] input={usage['input_tokens']} output={usage['output_tokens']} "
          f"cache={result['cache_read_tokens']} latency_ms={result['latency_ms']} "
          f"cost_usd={result['estimated_cost_usd']}")
    print(result["response"].text)


[full] input=110 output=181 cache=0 latency_ms=1494.9 cost_usd=0.000897
대출 관련 근거는 다음과 같습니다.

- **최대 대출 권수:** 회원 **1명당 최대 7권**  
  - **출처:** **[E01-p3]**

- **대출 기간:** **14일**  
  - **출처:** **[E01-p3]**

- **연장 조건:** **예약이 없으면 1회 7일 연장 가능**  
  - **출처:** **[E01-p3]**

참고로,
- **[E01-p2] 월요일 휴관**은 대출 규정이 아니라 **휴관일 정보**입니다.
- **[E01-p5] 회원은 최초 2시간 무료**는 **이용 요금/주차 등의 안내**로 보이며, 대출 조건과는 관련이 없습니다.

[selected] input=86 output=80 cache=0 latency_ms=1008.7 cost_usd=0.0004245
대출 조건은 다음과 같습니다.

- **대출 권수**: 회원 1명당 **최대 7권**
- **대출 기간**: **14일**
- **연장 조건**: **예약이 없는 경우에 한해 1회 7일 연장 가능**

**출처**: **[E01-p3]**

[compressed] input=82 output=84 cache=0 latency_ms=1057.0 cost_usd=0.0004395
대출 조건은 다음과 같습니다.

- **대출 권수**: 1명당 **최대 7권**
- **대출 기간**: **14일**
- **연장 조건**: **예약이 없을 때만 1회 7일 연장 가능**

**출처**: 제공해 주신 근거 **[E01-p3]**


### 필요한 정보가 보존됐는지 비교합니다

토큰 수가 작아도 대출 조건, 연장 예외, 출처가 빠지면 충분한 답이 아닙니다. 자동 검사는 문자열 존재 여부만 보여 주므로 답변 원문도 함께 읽습니다.

In [13]:
# 세 기준이 답변에 나타났는지 설계별로 확인합니다.
for name, result in design_results.items():
    answer = result["response"].text
    checks = {
        "대출 조건": "7권" in answer and "14일" in answer,
        "연장 예외": "연장" in answer,
        "출처": "E01" in answer,
    }
    result["checks"] = checks
    print(name, checks)

full {'대출 조건': True, '연장 예외': True, '출처': True}
selected {'대출 조건': True, '연장 예외': True, '출처': True}
compressed {'대출 조건': True, '연장 예외': True, '출처': True}


### 실행 기록을 보존합니다

Ch 7과 같이 비교 조건과 원문, 사용량, 지연, 예상 비용, 자동 확인을 정본과 실행별 사본에 저장합니다.

In [14]:
from datetime import datetime, timezone
import json
from pathlib import Path
run_at = datetime.now(timezone.utc)
session_id = run_at.strftime("%Y%m%dT%H%M%S%fZ")
results_for_log = []
for name, result in design_results.items():
    results_for_log.append({"design": name, "context": design_contexts[name],
                            "answer": result["response"].text, "usage": result["usage"],
                            "latency_ms": result["latency_ms"],
                            "cache_read_tokens": result["cache_read_tokens"],
                            "estimated_cost_usd": result["estimated_cost_usd"], "checks": result["checks"]})
record = {"run_id": f"ch08-context-{session_id}", "run_date": run_at.date().isoformat(),
          "source_chapter": "ch08", "source_record": {"path": str(ch07_relative), "sha256": ch07_sha256, "run_id": ch07_record["run_id"]}, "model": "openai:gpt-5.4-mini",
          "settings": {"max_tokens": 300, "pricing_date": ch07_record["settings"]["pricing_date"],
                       "pricing_usd_per_million": ch07_record["settings"]["pricing_usd_per_million"]},
          "question": question, "comparison_policy": "문자열 확인은 원문 정독 보조임", "results": results_for_log}
runs_dir = course_root / "labs/part2/ch08-context-engineering/runs"
record_path = runs_dir / "context-design-continuity.json"
session_path = runs_dir / "continuity" / session_id / record_path.name
session_path.parent.mkdir(parents=True, exist_ok=False)
payload = json.dumps(record, ensure_ascii=False, indent=2)
record_path.write_text(payload, encoding="utf-8")
session_path.write_text(payload, encoding="utf-8")
print(record_path.relative_to(course_root), "보존 사본:", session_path.relative_to(course_root))

labs/part2/ch08-context-engineering/runs/context-design-continuity.json 보존 사본: labs/part2/ch08-context-engineering/runs/continuity/20260909T175130932836Z/context-design-continuity.json


## 5. 목록 공개 수 한 값만 바꿉니다

전체 문서와 선택 결과는 그대로 두고 목록에서 보여 줄 항목 수만 `1`에서 `2`로 바꿉니다.

In [15]:
# 슬라이스 [:N]은 앞에서 N개 항목을 반환합니다.
base_limit = 1
changed_limit = 2
print("기본 목록:", catalog[:base_limit])
print("변경 목록:", catalog[:changed_limit])

기본 목록: [{'id': 'E01-p2', 'title': '이용 시간'}]
변경 목록: [{'id': 'E01-p2', 'title': '이용 시간'}, {'id': 'E01-p3', 'title': '대출 규정'}]


## 정리

목록, 선택, 원문 읽기를 순서대로 수행합니다. 토큰 감소와 필요한 조건의 보존 여부를 함께 비교합니다.

## 실패 재현: 질문과 다른 문서를 선택합니다

대출 질문에 주차 문서를 선택하면 입력은 짧아져도 필요한 대출 근거가 사라집니다.

In [16]:
# wrong_id는 질문과 관계없는 주차 문서 ID입니다.
# 반복문으로 실제 문서를 찾아 모델에 보낼 메시지를 만듭니다.
wrong_id = "E01-p5"
wrong_document = None
for document in documents:
    if document["id"] == wrong_id:
        wrong_document = document
        break

if wrong_document is None:
    raise ValueError("잘못 선택한 문서를 찾지 못했습니다.")

wrong_messages = [
    ("system", instruction),
    ("user", f"근거: [{wrong_document['id']}] {wrong_document['text']}\n질문: {question}"),
]

### 주차 문서를 모델에 전달해 답을 확인합니다

질문과 지시는 그대로 두고 선택한 문서만 `E01-p5`로 바꿉니다. 모델이 대출 조건의 근거를 받지 못한 상태에서 어떤 답을 하는지 읽습니다.

In [17]:
# 같은 모델을 실제 호출하고 선택한 문서와 답을 함께 출력합니다.
wrong_response = model.invoke(wrong_messages)
assert wrong_document["title"] == "주차 안내"
print("잘못 선택한 문서:", wrong_document["id"], wrong_document["title"])
print("모델 답:", wrong_response.text)

잘못 선택한 문서: E01-p5 주차 안내
모델 답: 아래 **근거 [E01-p5]**만으로는 **대출 권수, 대출 기간, 연장 조건**에 대한 정보가 없습니다. 확인 가능한 내용은 **“회원은 최초 2시간 무료”**라는 점뿐입니다.

### 확인된 근거
- **[E01-p5]** 회원은 최초 2시간 무료입니다.

### 질문별 답변
- **대출 권수:** 근거 없음
- **대출 기간:** 근거 없음
- **연장 조건:** 근거 없음
- **출처:** **[E01-p5]**

원하시면, 대출 권수/기간/연장 조건이 포함된 다른 근거를 주시면 그에 맞춰 정리해 드리겠습니다.


## 수정: 대출 규정 문서를 선택합니다

질문과 지시는 유지하고 대출 규정 `E01-p3`을 선택합니다. 같은 모델의 답에 대출 권수·기간·연장 조건이 나타나는지 확인합니다.

In [18]:
# 선택 ID만 대출 규정으로 고치고 같은 질문으로 다시 호출합니다.
fixed_id = "E01-p3"
fixed_document = None
for document in documents:
    if document["id"] == fixed_id:
        fixed_document = document
        break

if fixed_document is None:
    raise ValueError("수정한 문서를 찾지 못했습니다.")

assert fixed_document["title"] == "대출 규정"
fixed_messages = [
    ("system", instruction),
    ("user", f"근거: [{fixed_document['id']}] {fixed_document['text']}\n질문: {question}"),
]
fixed_response = model.invoke(fixed_messages)
assert "7권" in fixed_response.text and "14일" in fixed_response.text
assert "연장" in fixed_response.text
print("수정 후 선택:", fixed_document["id"], fixed_document["title"])
print("모델 답:", fixed_response.text)

수정 후 선택: E01-p3 대출 규정
모델 답: 근거는 다음과 같습니다.

- **대출 권수:** 회원 1명당 **최대 7권**
- **대출 기간:** **14일**
- **연장 조건:** **예약이 없을 경우**에 한해 **1회 7일 연장 가능**

**출처:** `[E01-p3]`
